## Machine Learning for Thesis

1. Within-language
    1.1 Family 1 - traditional ML (SVM, LR, XGBoost — within-language only, TF-IDF input)
    1.2 Family 2 - (mDeBERTa — within-language baseline)

2. Transfer Learning
    mDeBERTa - cross-lingual transfer
    (EN -> CN)
    (EN -> JP)
    (CN -> JP)


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import stopwordsiso
from stopwordsiso import stopwords
import joblib
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import Dataset


d:\Anaconda\envs\dataviz\Lib\site-packages\stopwordsiso\_core.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
d:\Anaconda\envs\dataviz\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import logging
logging.basicConfig(filename='training_log.txt', level=logging.INFO)


Examine environment check GPU

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)                  # CUDA version
print(torch.__version__) 
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")



True
NVIDIA GeForce RTX 4080 Laptop GPU
12.4
2.6.0+cu124
12.878086144 GB


## Load saved models

In [4]:
# # Reload vectorizers and models
# tfidf_E = joblib.load('tfidf_E.pkl')
# model_SVM_E = joblib.load('model_SVM_E.pkl')

# # Reload results summary
# with open('ml_results_summary.json', 'r') as f:
#     summary = json.load(f)


## Load tokenized ML training & Testing data

In [5]:
# X_train_E, X_test_E, y_train_E, y_test_E = joblib.load('models/split_EN.pkl')
# X_train_C, X_test_C, y_train_C, y_test_C = joblib.load('models/split_CN.pkl')
# X_train_J, X_test_J, y_train_J, y_test_J = joblib.load('models/split_JP.pkl')

## Load raw ML training data

In [6]:
# X_train_E_raw, X_test_E_raw = joblib.load('models/split_EN_raw.pkl')
# X_train_C_raw, X_test_C_raw = joblib.load('models/split_CN_raw.pkl')
# X_train_J_raw, X_test_J_raw = joblib.load('models/split_JP_raw.pkl')


## Check for duplicates

In [7]:
# # Check for duplicates between train and test
# import pandas as pd
# train_series = pd.Series(X_train_E_raw)
# test_series = pd.Series(X_test_E_raw)
# overlap = test_series.isin(train_series).sum()
# print(f"Exact duplicates between EN train and test: {overlap}")

# # Check label distribution
# import numpy as np
# print(f"EN train label dist: {np.bincount(y_train_E)}")
# print(f"EN test label dist:  {np.bincount(y_test_E)}")

In [8]:
# for name, X_tr, X_te in [('CN', X_train_C_raw, X_test_C_raw), 
#                            ('JP', X_train_J_raw, X_test_J_raw)]:
#     overlap = pd.Series(X_te).isin(pd.Series(X_tr)).sum()
#     print(f"{name} duplicates: {overlap}")

## data preparation, remove stopwords

In [9]:
# load data

en = pd.read_csv('data/en_clean.csv')
cn = pd.read_csv('data/cn_clean.csv')
jp = pd.read_csv('data/jp_clean.csv')

en.head()

,label,merged_text,cleaned_text,tokenized,cleaned_len,cleaned_empty,token_len,token_empty
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,law enforcement on high alert following threat...,"['law', 'enforcement', 'on', 'high', 'alert', ...",5180,False,931,False
1,0,Did they post their votes for Hillary already?,did they post their votes for hillary already?,"['did', 'they', 'post', 'their', 'votes', 'for...",46,False,8,False
2,0,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,unbelievable! obama’s attorney general says mo...,"['unbelievable', 'obama', 's', 'attorney', 'ge...",354,False,54,False
3,1,"Bobby Jindal, raised Hindu, uses story of Chri...","bobby jindal, raised hindu, uses story of chri...","['bobby', 'jindal', 'raised', 'hindu', 'uses',...",8116,False,1358,False
4,0,SATAN 2: Russia unvelis an image of its terrif...,satan 2: russia unvelis an image of its terrif...,"['satan', '2', 'russia', 'unvelis', 'an', 'ima...",2012,False,358,False


## DEDUP

In [10]:
en = en.drop_duplicates(subset='merged_text').reset_index(drop=True)
cn = cn.drop_duplicates(subset='merged_text').reset_index(drop=True)
jp = jp.drop_duplicates(subset='merged_text').reset_index(drop=True)

print(f"EN: {len(en)} samples after dedup")
print(f"CN: {len(cn)} samples after dedup")
print(f"JP: {len(jp)} samples after dedup")

EN: 63672 samples after dedup
CN: 15108 samples after dedup
JP: 13310 samples after dedup


In [11]:
def str_to_list(df): # converts the tokenized column back to a list 
    import ast
    df = df.copy()
    df['tokenized'] = df['tokenized'].apply(ast.literal_eval)
    return df

In [12]:
en = str_to_list(en)
cn = str_to_list(cn)
jp = str_to_list(jp)

In [13]:
en['tokenized'].dtype

dtype('O')

In [14]:
def rem_sw(df, lang):
    sw = set(stopwords(lang))  # faster lookup
    
    def remove(tokens):
        return [word for word in tokens if word not in sw]
    
    df = df.copy()
    df['token_sw'] = df['tokenized'].apply(remove)
    return df


In [15]:
en = rem_sw(en, 'en')
cn = rem_sw(cn, 'zh')
jp = rem_sw(jp, 'ja')

In [1]:
en.head()

NameError: name 'en' is not defined

In [17]:
cn.head()

,label,merged_text,cleaned_text,tokenized,cleaned_len,cleaned_empty,token_len,token_empty,token_sw
0,0,青少年妈妈明星珍妮尔·埃文斯的婚纱在这里售价2999美元 2023年3月23日 · 青少年妈...,青少年妈妈明星珍妮尔埃文斯的婚纱在这里售价2999美元2023年3月23日青少年妈妈明星Je...,"[青少年, 妈妈, 明星, 珍妮, 尔, 埃文斯, 的, 婚纱, 在, 这里, 售价, 29...",117,False,59,False,"[青少年, 妈妈, 明星, 珍妮, 埃文斯, 婚纱, 售价, 2999, 美元, 2023,..."
1,0,凯莉·詹纳拒绝就《凯莉的一生》讨论泰加 2020年2月23日 · 2018年詹纳才20岁（她...,凯莉詹纳拒绝就凯莉的一生讨论泰加2020年2月23日2018年詹纳才20岁她在8月满21岁同...,"[凯莉, 詹纳, 拒绝, 就, 凯莉, 的, 一生, 讨论, 泰加, 2020, 年, 2,...",94,False,57,False,"[凯莉, 詹纳, 拒绝, 凯莉, 一生, 讨论, 泰加, 2020, 2, 23, 2018..."
2,0,奎因帕金斯 2023年3月30日 · 虎扑03月30日讯 今日步行者官方宣布，球队正式裁掉后...,奎因帕金斯2023年3月30日虎扑03月30日讯今日步行者官方宣布球队正式裁掉后卫特雷夫林奎...,"[奎因, 帕金斯, 2023, 年, 3, 月, 30, 日虎, 扑, 03, 月, 30,...",82,False,41,False,"[奎因, 帕金斯, 2023, 3, 30, 日虎, 扑, 03, 30, 日讯, 今日, ..."
3,0,席琳·迪翁向拉斯维加斯枪击案受害者捐赠音乐会收益 2021年11月10日 · 今年10月下旬...,席琳迪翁向拉斯维加斯枪击案受害者捐赠音乐会收益2021年11月10日今年10月下旬席琳迪翁亲...,"[席琳迪翁, 向, 拉斯维加斯, 枪击案, 受害者, 捐赠, 音乐会, 收益, 2021, ...",100,False,51,False,"[席琳迪翁, 拉斯维加斯, 枪击案, 受害者, 捐赠, 音乐会, 收益, 2021, 11,..."
4,0,Chris Evans、Millie Bobby Brown、Snoop Dogg等明星为被...,ChrisEvansMillieBobbyBrownSnoopDogg等明星为被欺负的学生K...,"[ChrisEvansMillieBobbyBrownSnoopDogg, 等, 明星, 为...",140,False,12,False,"[ChrisEvansMillieBobbyBrownSnoopDogg, 明星, 欺负, ..."


In [18]:
jp.head()

,label,merged_text,cleaned_text,tokenized,cleaned_len,cleaned_empty,token_len,token_empty,token_sw
0,0,朝日新聞など各社の報道によれば、宅配便最大手「ヤマト運輸」が日本郵政公社を相手取り、大手コン...,朝日新聞など各社の報道によれば宅配便最大手ヤマト運輸が日本郵政公社を相手取り大手コンビニエン...,"[朝日新聞, など, 各社, の, 報道, に, よれ, ば, 宅配便, 最大手, ヤマト運...",497,False,250,False,"[朝日新聞, 各社, 報道, よれ, 宅配便, 最大手, ヤマト運輸, 日本郵政, 公社, ..."
1,1,11月5日の各社報道によると、諫早湾干拓事業は諫早海人（諫早湾の「海」）に囲まれる大洋に位置...,11月5日の各社報道によると諫早湾干拓事業は諫早海人諫早湾の海に囲まれる大洋に位置することか...,"[11, 月, 5, 日, の, 各社, 報道, に, よる, と, 諫早湾, 干拓, 事業...",360,False,207,False,"[11, 月, 5, 日, 各社, 報道, よる, 諫早湾, 干拓, 事業, 諫早, 海人,..."
2,1,産経新聞、中日新聞によると、2004年から2005年まで、この大会による3年おきの開催を、2...,産経新聞中日新聞によると2004年から2005年までこの大会による3年おきの開催を2006年...,"[産経新聞, 中日, 新聞, に, よる, と, 2004, 年, から, 2005, 年,...",231,False,146,False,"[産経新聞, 中日, 新聞, よる, 2004, 年, 2005, 年, 大会, よる, 3..."
3,1,開催地のリオデジャネイロ市に対して、大会期間中のリオデジャネイロオリンピックに関する公式発表...,開催地のリオデジャネイロ市に対して大会期間中のリオデジャネイロオリンピックに関する公式発表は...,"[開催地, の, リオデジャネイロ, 市, に, 対し, て, 大会, 期間中, の, リオ...",626,False,249,False,"[開催地, リオデジャネイロ, 市, 対し, 大会, 期間中, リオデジャネイロ, オリンピ..."
4,1,毎日新聞・時事通信によると、2006年2月13日には、グッドウィル・グッゲンハイム・アン・ハ...,毎日新聞時事通信によると2006年2月13日にはグッドウィルグッゲンハイムアンハルクを経営す...,"[毎日, 新聞, 時事, 通信, に, よる, と, 2006, 年, 2, 月, 13, ...",224,False,108,False,"[毎日, 新聞, 時事, 通信, よる, 2006, 年, 2, 月, 13, 日, グッド..."


In [19]:
en_ml = en[['label', 'token_sw']]
cn_ml = cn[['label', 'token_sw']]
jp_ml = jp[['label', 'token_sw']]

In [20]:
en_ml.head()

,label,token_sw
0,0,"[law, enforcement, alert, threats, cops, white..."
1,0,"[post, votes, hillary]"
2,0,"[unbelievable, obama, attorney, charlotte, rio..."
3,1,"[bobby, jindal, raised, hindu, story, christia..."
4,0,"[satan, 2, russia, unvelis, image, terrifying,..."


## Train test split

In [21]:
def split_data(df):
    X_train, X_test, y_train, y_test = train_test_split(
        df['token_sw'], 
        df['label'], 
        test_size=0.2, 
        random_state=42, 
        stratify=df['label']) # ensures train and test splits have the same real/fake ratio
    return X_train, X_test, y_train, y_test

In [22]:
X_train_E, X_test_E, y_train_E, y_test_E = split_data(en_ml)
X_train_C, X_test_C, y_train_C, y_test_C = split_data(cn_ml)
X_train_J, X_test_J, y_train_J, y_test_J = split_data(jp_ml)



## Save tokenized ML models

In [23]:
# --- Save train&test vectorizers ---
joblib.dump((X_train_E, X_test_E, y_train_E, y_test_E), 'models/split_EN.pkl')
joblib.dump((X_train_C, X_test_C, y_train_C, y_test_C), 'models/split_CN.pkl')
joblib.dump((X_train_J, X_test_J, y_train_J, y_test_J), 'models/split_JP.pkl')



print("Splits saved.")

Splits saved.


## TF-IDF for all three languages

In [24]:
# tfIDF 

from sklearn.feature_extraction.text import TfidfVectorizer

# --- English ---
tfidf_E = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_E_tfidf = tfidf_E.fit_transform(X_train_E.apply(lambda x: ' '.join(x) if isinstance(x, list) else x))
X_test_E_tfidf  = tfidf_E.transform(X_test_E.apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

# --- Chinese ---
tfidf_C = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_C_tfidf = tfidf_C.fit_transform(X_train_C.apply(lambda x: ' '.join(x) if isinstance(x, list) else x))
X_test_C_tfidf  = tfidf_C.transform(X_test_C.apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

# --- Japanese ---
tfidf_J = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_J_tfidf = tfidf_J.fit_transform(X_train_J.apply(lambda x: ' '.join(x) if isinstance(x, list) else x))
X_test_J_tfidf  = tfidf_J.transform(X_test_J.apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

## ML Training

In [25]:
# --- Hyperparameter grids ---
# --- Hyperparameter grids ---
lr_params  = {'C': [0.01, 0.1, 1, 10]}
svm_params = {'C': [0.01, 0.1, 1, 10]}
xgb_params = {'n_estimators': [100, 200],
               'max_depth': [3, 6],
               'learning_rate': [0.1]}  # removed 0.01 — never won, just adds time

def train_evaluate(X_train, X_test, y_train, y_test, lang):
    results = {}

    # calculate class imbalance ratio for XGB
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = round(neg / pos, 2)  # scale_pos_weight = real/fake ratio

    models = {
    'LR':  GridSearchCV(LogisticRegression(max_iter=1000), lr_params, 
                        cv=5, scoring='f1_macro', n_jobs=2),   # was -1
    'SVM': GridSearchCV(LinearSVC(max_iter=2000), svm_params, 
                        cv=5, scoring='f1_macro', n_jobs=2),   # was -1
    'XGB': GridSearchCV(
                XGBClassifier(eval_metric='logloss', verbosity=0,
                              n_jobs=1,
                              scale_pos_weight=spw),
                xgb_params, cv=5, scoring='f1_macro', n_jobs=1)  # was -1, XGB especially dangerous
    }

    for name, model in models.items():
        print(f"Training {name} on {lang}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        print(f"\n{lang} - {name} (best params: {model.best_params_})")
        print(classification_report(y_test, y_pred))
        results[name] = {
            'best_params': model.best_params_,
            'macro_f1':    report['macro avg']['f1-score'],
            'accuracy':    report['accuracy'],
            'full_report': report,
            'model':       model
        }

    return results

# --- Run ---
results_E = train_evaluate(X_train_E_tfidf, X_test_E_tfidf, y_train_E, y_test_E, 'English')
results_C = train_evaluate(X_train_C_tfidf, X_test_C_tfidf, y_train_C, y_test_C, 'Chinese')
results_J = train_evaluate(X_train_J_tfidf, X_test_J_tfidf, y_train_J, y_test_J, 'Japanese')



Training LR on English...

English - LR (best params: {'C': 10})
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      5776
           1       0.96      0.97      0.96      6959

    accuracy                           0.96     12735
   macro avg       0.96      0.96      0.96     12735
weighted avg       0.96      0.96      0.96     12735

Training SVM on English...

English - SVM (best params: {'C': 1})
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      5776
           1       0.96      0.97      0.96      6959

    accuracy                           0.96     12735
   macro avg       0.96      0.96      0.96     12735
weighted avg       0.96      0.96      0.96     12735

Training XGB on English...

English - XGB (best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200})
              precision    recall  f1-score   support

           0       0.94      0.97      0.96

## Save model results

In [26]:
# --- Save TF-IDF vectorizers ---
joblib.dump(tfidf_E, 'tfidf_E.pkl')
joblib.dump(tfidf_C, 'tfidf_C.pkl')
joblib.dump(tfidf_J, 'tfidf_J.pkl')

# --- Save best models ---
joblib.dump(results_E['LR']['model'],  'model_LR_E.pkl')
joblib.dump(results_E['SVM']['model'], 'model_SVM_E.pkl')
joblib.dump(results_E['XGB']['model'], 'model_XGB_E.pkl')

joblib.dump(results_C['LR']['model'],  'model_LR_C.pkl')
joblib.dump(results_C['SVM']['model'], 'model_SVM_C.pkl')
joblib.dump(results_C['XGB']['model'], 'model_XGB_C.pkl')

joblib.dump(results_J['LR']['model'],  'model_LR_J.pkl')
joblib.dump(results_J['SVM']['model'], 'model_SVM_J.pkl')
joblib.dump(results_J['XGB']['model'], 'model_XGB_J.pkl')

# --- Save results summary (metrics + best params) ---
summary = {}
for lang, results in [('EN', results_E), ('CN', results_C), ('JP', results_J)]:
    summary[lang] = {}
    for model_name, info in results.items():
        summary[lang][model_name] = {
            'best_params': info['best_params'],
            'macro_f1':    info['macro_f1'],
            'accuracy':    info['accuracy'],
            'full_report': info['full_report']
        }

with open('ml_results_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)

print("All models and results saved.")


All models and results saved.


## DL - baseline 

### train test split on raw text column

In [27]:
from sklearn.model_selection import train_test_split

# Assuming your original df has a 'text' column (raw) and 'label'
# Recreate the same split using the same random_state=42

X_train_E_raw, X_test_E_raw, _, _ = train_test_split(
    en['merged_text'], en['label'],
    test_size=0.2, random_state=42, stratify=en['label']
)

X_train_C_raw, X_test_C_raw, _, _ = train_test_split(
    cn['merged_text'], cn['label'],
    test_size=0.2, random_state=42, stratify=cn['label']
)

X_train_J_raw, X_test_J_raw, _, _ = train_test_split(
    jp['merged_text'], jp['label'],
    test_size=0.2, random_state=42, stratify=jp['label']
)

## Save DL training and test sets



In [28]:
# --- Save train&test vectorizers ---
joblib.dump((X_train_E_raw, X_test_E_raw), 'models/split_EN_raw.pkl')
joblib.dump((X_train_C_raw, X_test_C_raw), 'models/split_CN_raw.pkl')
joblib.dump((X_train_J_raw, X_test_J_raw), 'models/split_JP_raw.pkl')



['models/split_JP_raw.pkl']

### Finetuning

In [29]:
# --- Check GPU ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Dataset class ---
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# --- Metrics function ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

# --- Training function ---
def train_mdeberta(X_train, X_test, y_train, y_test, lang, save_dir):
    print(f"\n--- Training mDeBERTa on {lang} ---")

    train_dataset = FakeNewsDataset(X_train, y_train, tokenizer)
    test_dataset  = FakeNewsDataset(X_test,  y_test,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    training_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        eval_strategy='epoch',        # renamed from evaluation_strategy
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        fp16=True,
        bf16=False,             
        seed=42
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # --- Evaluate ---
    preds_output = trainer.predict(test_dataset)
    preds = np.argmax(preds_output.predictions, axis=1)
    print(f"\n{lang} - mDeBERTa")
    print(classification_report(y_test, preds))

    report = classification_report(y_test, preds, output_dict=True)

    # --- Save model ---
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Model saved to {save_dir}")

    return trainer, preds, report

# --- Run within-language ---
trainer_E, preds_E, report_E = train_mdeberta(X_train_E_raw, X_test_E_raw, y_train_E, y_test_E,
                                     lang='English', save_dir='models/xlmr_EN')

trainer_C, preds_C, report_C = train_mdeberta(X_train_C_raw, X_test_C_raw, y_train_C, y_test_C,
                                     lang='Chinese', save_dir='models/xlmr_CN')

trainer_J, preds_J, report_J = train_mdeberta(X_train_J_raw, X_test_J_raw, y_train_J, y_test_J,
                                     lang='Japanese', save_dir='models/xlmr_JP')

# Save monolingual XLM-R results
monolingual_results = {
    'EN': {'macro_f1': report_E['macro avg']['f1-score'], 'accuracy': report_E['accuracy'], 'full_report': report_E},
    'CN': {'macro_f1': report_C['macro avg']['f1-score'], 'accuracy': report_C['accuracy'], 'full_report': report_C},
    'JP': {'macro_f1': report_J['macro avg']['f1-score'], 'accuracy': report_J['accuracy'], 'full_report': report_J},
}
with open('results/monolingual_xlmr_results.json', 'w') as f:
    json.dump(monolingual_results, f, indent=4)
print('Monolingual XLM-R results saved.')


Using device: cuda



--- Training mDeBERTa on English ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6712.63it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.092699,0.041674,0.990087
2,0.058501,0.051359,0.990960
3,0.034170,0.032309,0.993978
4,0.013535,0.064058,0.989847
5,0.008988,0.038567,0.994295


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

...

Epoch,Training Loss,Validation Loss,Macro F1
1,0.693309,0.290169,0.878393
2,0.544209,0.261345,0.861973
3,0.477719,0.255195,0.888832
4,0.406456,0.249162,0.882842
5,0.381764,0.299250,0.888282


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la


Japanese - mDeBERTa
              precision    recall  f1-score   support

           0       0.76      0.96      0.85       737
           1       0.98      0.89      0.93      1925

    accuracy                           0.90      2662
   macro avg       0.87      0.92      0.89      2662
weighted avg       0.92      0.90      0.91      2662



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Model saved to models/xlmr_JP
Monolingual XLM-R results saved.


In [30]:
# Check for NaN in raw text
print("EN NaN:", X_train_E_raw.isna().sum())
print("Labels NaN:", y_train_E.isna().sum())

# Check label types and values
print("Label dtype:", y_train_E.dtype)
print("Unique labels:", y_train_E.unique())

# Check a few samples
print(X_train_E_raw.head())

EN NaN: 0
Labels NaN: 0
Label dtype: int64
Unique labels: [1 0]
59110    Uber CEO quits Trump's business advisory group...
14753    Senator Warren calls on Fed to remove Wells Fa...
58598    Man Wearing ‘Jewmerica’ T-Shirt Never Dreamed ...
39463     Unlike Hillary, Cruz’s Campaign May Have ACTU...
56256     The Weather Channel Just SLAMMED Breitbart Fo...
Name: merged_text, dtype: str


In [31]:
print(torch.cuda.is_available())
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")


True
10.048301056 GB allocated
12.71922688 GB reserved


In [32]:
import shutil
shutil.rmtree('models/mdeberta_EN', ignore_errors=True)
shutil.rmtree('models/mdeberta_CN', ignore_errors=True)
shutil.rmtree('models/mdeberta_JP', ignore_errors=True)



## Transfer Learning

In [33]:
import os
os.makedirs('results', exist_ok=True)

In [34]:
def cross_lingual_eval(model_dir, X_test, y_test, source_lang, target_lang):
    print(f"\n--- Cross-lingual Transfer: {source_lang} → {target_lang} ---")
    
    # Load the saved fine-tuned model and tokenizer
    tokenizer_cl = AutoTokenizer.from_pretrained(model_dir)
    model_cl = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model_cl.to(device)
    model_cl.eval()
    
    # Tokenize target language test set
    test_dataset = FakeNewsDataset(X_test, y_test, tokenizer_cl)
    
    # Run predictions
    all_preds = []
    all_labels = []
    
    from torch.utils.data import DataLoader
    loader = DataLoader(test_dataset, batch_size=32)
    
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels']
            
            outputs = model_cl(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    print(f"\n{source_lang} → {target_lang} - XLM-R")
    print(classification_report(all_labels, all_preds))
    
    report = classification_report(all_labels, all_preds, output_dict=True)
    return {
        'source': source_lang,
        'target': target_lang,
        'macro_f1': report['macro avg']['f1-score'],
        'accuracy': report['accuracy'],
        'full_report': report
    }

# --- Run cross-lingual transfer ---
results_cross = []

# EN as source
results_cross.append(cross_lingual_eval('models/xlmr_EN', X_test_C_raw, y_test_C,
                                         source_lang='EN', target_lang='CN'))
results_cross.append(cross_lingual_eval('models/xlmr_EN', X_test_J_raw, y_test_J,
                                         source_lang='EN', target_lang='JP'))

# CN as source
results_cross.append(cross_lingual_eval('models/xlmr_CN', X_test_J_raw, y_test_J,
                                         source_lang='CN', target_lang='JP'))
results_cross.append(cross_lingual_eval('models/xlmr_CN', X_test_E_raw, y_test_E,
                                         source_lang='CN', target_lang='EN'))

# --- Save results ---
import json
with open('results/cross_lingual_results.json', 'w') as f:
    json.dump(results_cross, f, indent=4)

print("\nCross-lingual results saved.")



--- Cross-lingual Transfer: EN → CN ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5959.12it/s]



EN → CN - XLM-R
              precision    recall  f1-score   support

           0       0.36      0.98      0.53      1089
           1       0.41      0.01      0.01      1933

    accuracy                           0.36      3022
   macro avg       0.39      0.50      0.27      3022
weighted avg       0.39      0.36      0.20      3022


--- Cross-lingual Transfer: EN → JP ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11435.60it/s]



EN → JP - XLM-R
              precision    recall  f1-score   support

           0       0.29      0.67      0.40       737
           1       0.75      0.38      0.50      1925

    accuracy                           0.46      2662
   macro avg       0.52      0.52      0.45      2662
weighted avg       0.62      0.46      0.47      2662


--- Cross-lingual Transfer: CN → JP ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6054.04it/s]



CN → JP - XLM-R
              precision    recall  f1-score   support

           0       0.29      0.61      0.39       737
           1       0.74      0.42      0.53      1925

    accuracy                           0.47      2662
   macro avg       0.51      0.51      0.46      2662
weighted avg       0.61      0.47      0.49      2662


--- Cross-lingual Transfer: CN → EN ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10832.84it/s]



CN → EN - XLM-R
              precision    recall  f1-score   support

           0       0.16      0.11      0.13      5776
           1       0.42      0.53      0.47      6959

    accuracy                           0.34     12735
   macro avg       0.29      0.32      0.30     12735
weighted avg       0.30      0.34      0.31     12735


Cross-lingual results saved.


In [35]:
def finetune_transfer(source_model_dir, X_train_target, X_test_target, 
                       y_train_target, y_test_target,
                       source_lang, target_lang, 
                       save_dir, subset=0.1):
    
    print(f"\n--- Fine-tune Transfer: {source_lang} → {target_lang} (subset={subset}) ---")
    
    # Take small subset of target training data
    X_sub, _, y_sub, _ = train_test_split(
        X_train_target, y_train_target,
        train_size=subset,
        random_state=42,
        stratify=y_train_target
    )
    print(f"Fine-tuning on {len(X_sub)} target language samples ({subset*100:.0f}% of {target_lang} train)")
    
    # Load source fine-tuned model
    tokenizer_ft = AutoTokenizer.from_pretrained(source_model_dir)
    model_ft = AutoModelForSequenceClassification.from_pretrained(source_model_dir)
    
    # Create datasets
    train_dataset = FakeNewsDataset(X_sub, y_sub, tokenizer_ft)
    test_dataset  = FakeNewsDataset(X_test_target, y_test_target, tokenizer_ft)
    
    training_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=3,              # fewer epochs — already partially trained
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=1e-5,              # lower LR — fine-tuning on top of fine-tuning
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        fp16=True,
        bf16=False,
        seed=42
    )
    
    trainer = Trainer(
        model=model_ft,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    # Evaluate
    preds_output = trainer.predict(test_dataset)
    preds = np.argmax(preds_output.predictions, axis=1)
    print(f"\n{source_lang} → {target_lang} fine-tuned - XLM-R")
    print(classification_report(y_test_target, preds))
    
    report = classification_report(y_test_target, preds, output_dict=True)
    
    # Save
    trainer.save_model(save_dir)
    tokenizer_ft.save_pretrained(save_dir)
    
    return {
        'source': source_lang,
        'target': target_lang,
        'setup': 'fine-tune-transfer',
        'subset': subset,
        'macro_f1': report['macro avg']['f1-score'],
        'accuracy': report['accuracy'],
        'full_report': report
    }

# Make sure output dirs exist
os.makedirs('results', exist_ok=True)
os.makedirs('models/xlmr_EN_to_CN', exist_ok=True)
os.makedirs('models/xlmr_EN_to_JP', exist_ok=True)
os.makedirs('models/xlmr_CN_to_JP', exist_ok=True)

results_finetune = []

# EN → CN
results_finetune.append(finetune_transfer(
    source_model_dir='models/xlmr_EN',
    X_train_target=X_train_C_raw, X_test_target=X_test_C_raw,
    y_train_target=y_train_C,     y_test_target=y_test_C,
    source_lang='EN', target_lang='CN',
    save_dir='models/xlmr_EN_to_CN'
))

# EN → JP
results_finetune.append(finetune_transfer(
    source_model_dir='models/xlmr_EN',
    X_train_target=X_train_J_raw, X_test_target=X_test_J_raw,
    y_train_target=y_train_J,     y_test_target=y_test_J,
    source_lang='EN', target_lang='JP',
    save_dir='models/xlmr_EN_to_JP'
))

# CN → JP
results_finetune.append(finetune_transfer(
    source_model_dir='models/xlmr_CN',
    X_train_target=X_train_J_raw, X_test_target=X_test_J_raw,
    y_train_target=y_train_J,     y_test_target=y_test_J,
    source_lang='CN', target_lang='JP',
    save_dir='models/xlmr_CN_to_JP'
))

# Save results
import json
with open('results/finetune_transfer_results.json', 'w') as f:
    json.dump(results_finetune, f, indent=4)

print("\nFine-tune transfer results saved.")



--- Fine-tune Transfer: EN → CN (subset=0.1) ---
Fine-tuning on 1208 target language samples (10% of CN train)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6731.84it/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.572828,0.693205
2,No log,0.524616,0.716103
3,No log,0.500672,0.755513


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la


EN → CN fine-tuned - XLM-R
              precision    recall  f1-score   support

           0       0.76      0.60      0.67      1089
           1       0.80      0.89      0.84      1933

    accuracy                           0.79      3022
   macro avg       0.78      0.75      0.76      3022
weighted avg       0.78      0.79      0.78      3022



...

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.496669,0.733320
2,No log,0.430460,0.795191
3,No log,0.387041,0.804037


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la


CN → JP fine-tuned - XLM-R
              precision    recall  f1-score   support

           0       0.67      0.79      0.73       737
           1       0.91      0.85      0.88      1925

    accuracy                           0.84      2662
   macro avg       0.79      0.82      0.80      2662
weighted avg       0.85      0.84      0.84      2662



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]



Fine-tune transfer results saved.
